# LangGraph

<a name="top"></a>

1. [Workflows vs Agents](#workflows-and-agents)
2. [Основные строительные блоки графа](#building-blocks)
3. [Архитектурные паттерны (композиция LLM-вызовов)](#architecture-patterns)
   - [Prompt Chaining](#prompt-chaining)
   - [Parallelization](#parallelization)
   - [Routing](#routing)
   - [Orchestrator-Worker](#orchestrator-worker)
   - [Evaluator-Optimizer](#evaluator-optimizer)

### Module 2: Управление State (Persistence)
- [1. Основные понятия](#basic)
- [2. Работа с чекпоинтами](#checkpoints-work)
- [3. Воспроизведение (Replay)](#replay)
- [4. Ручное обновление State (update_state)](#update-state)
- [5. Ручное обновление State (as_node)](#as_node)

### Module 3: Продвинутое управление State и устойчивость выполнения (Durability)
- [1. Checkpoint и CheckpointInterface](#module3-checkpoint-interface)
- [2. Ключевые возможности на базе чекпоинтов](#module3-key-features)
  - [Human-in-the-loop (HITL)](#module3-hitl)
  - [Timetravel (путешествие во времени)](#module3-timetravel)
  - [Отказоустойчивость (Fault-tolerance) и ожидающие записи (Pending writes)](#module3-fault-tolerance)
- [3. Durable Execution (Устойчивое выполнение)](#module3-durable-execution)
  - [Как это работает в LangGraph](#module3-how-it-works)
  - [Обёртывание недетерминированных операций в @task](#module3-task)
  - [Важный нюанс при возобновлении](#module3-resume-nuance)
- [4. Durability Modes (режимы сохранения)](#module3-durability-modes)
- [5. Использование Task в узлах (более подробно)](#module3-task-details)
- [6. Возобновление workflows (Resuming Workflows)](#module3-resuming)
  - [Пауза и возобновление через Interrupts и Commands](#module3-interrupts-commands)
  - [Восстановление после ошибок через указание checkpoint_id](#module3-error-recovery)

### Module 4: Управление памятью в LangGraph
- [1. Store – абстракция долговременной памяти](#module4-store)
  - [Основные понятия Store](#module4-store-concepts)
  - [Пример работы с InMemoryStore](#module4-store-example)
  - [Поиск и получение данных](#module4-store-search)
  - [Семантический поиск (Semantic Search)](#module4-semantic-search)
  - [Добавление Store в граф](#module4-store-in-graph)
- [2. Short-term memory (кратковременная память) на базе чекпоинтов](#module4-short-term)
- [3. Long-term memory vs Short-term memory (таблица)](#module4-comparison)
- [4. Управление длиной контекста (Context Management)](#module4-context-management)
  - [Trim (обрезание)](#module4-trim)
  - [Delete (удаление)](#module4-delete)
  - [Summarize (суммаризация)](#module4-summarize)
  - [Manage Checkpoints (управление чекпоинтами)](#module4-manage-checkpoints)
- [5. Миграции баз данных](#module4-migrations)

### Module 5: Потоковая передача и подграфы (Streaming & Subgraphs)
- [1. Streaming (потоковая передача)](#streaming)
  - [Режимы стриминга (`stream_mode`)](#streaming-modes)
- [2. Subgraph (подграфы)](#subgraph)
  - [Как это работает](#subgraph-how)
  - [Пример](#subgraph-example)
  - [Преимущества subgraph](#subgraph-benefits)

### Module 6: Мультиагентные системы (Multi-Agent Systems)
- [1. Базовые принципы координации](#module6-basic-principles)
- [2. Паттерны мультиагентного взаимодействия](#module6-patterns)
  - [Supervisor (Агент-Супервизор)](#module6-supervisor)
  - [Hierarchical Teams (Иерархические команды)](#module6-hierarchical-teams)
  - [Multi-Agent Collaboration (Равноправное сотрудничество)](#module6-collaboration)
- [3. Коммуникация между агентами](#module6-communication)

### Module 7: Наблюдаемость и тестирование (Observability & Testing)
- [1. Observability (Наблюдаемость) с OpenTelemetry](#module7-observability)
- [2. Тестирование и оценка качества (Evaluation, Evals)](#module7-evaluation)

### Module 8: Инструменты и Production (Tooling & Deployment)
- [1. Prebuilt Components (Готовые компоненты)](#module8-prebuilt)
- [2. Деплой в Production (LangGraph Platform)](#module8-langgraph-platform)
- [3. Альтернативный деплой: Самостоятельно на AWS](#module8-aws-deploy)

- [Уровни владения LangGraph](#lg-skills)

### 1. Workflows vs Agents <a id="workflows-and-agents"></a>

- **Workflows** – жёстко заданные пути выполнения (направленные графы). Порядок узлов и переходы между ними определены заранее. Подходит для предсказуемых задач, где логика известна.
- **Agents** – автономные системы, которые сами решают, когда и какие инструменты вызывать. 
В LangGraph агент реализуется как цикл: Node «агент» принимает решение, вызывает инструменты, обрабатывает результаты и повторяет до завершения.

### 2. Основные строительные блоки графа <a id="building-blocks"></a>

- **State** – общий словарь (или типизированный объект), который проходит через все узлы. Определяет структуру данных, которые хранятся и обновляются.
- **Node** – функция, которая принимает текущий State и возвращает обновление (часть нового State).
- **Edge** – связи между узлами.
- **Conditional Edge** – переход, который выбирает следующий Node на основе текущего State (функция-маршрутизатор).

#### Импорты и настройка

```python
# Базовые импорты
from typing import TypedDict, Annotated, List, Literal, Optional
import operator
import uuid

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command, interrupt, StreamWriter
from langgraph.task import task

# Для новых StateSchema (LangGraph 1.0+)
from langgraph.graph import StateSchema, ReducedValue, MessagesValue, UntrackedValue

# Для prebuilt компонентов
from langgraph.prebuilt import create_react_agent, ToolNode, HumanInterruptNode

# LangChain сообщения
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI  # или другая модель
```

### 3. Архитектурные паттерны (композиция LLM-вызовов) <a id="architecture-patterns"></a>


#### Prompt Chaining <a id="prompt-chaining"></a>
- **Суть**: результат одного вызова LLM передаётся в следующий. Каждый шаг может использовать свой промпт и модель.
- **Пример**: генерация плана → написание текста по плану → проверка грамматики.
- **В LangGraph**: последовательные узлы, соединённые простыми edges.

#### Parallelization <a id="parallelization"></a>
- **Суть**: несколько worker Nodes запускаются параллельно, каждый обрабатывает входные данные (или часть данных), затем их результаты агрегируются.
- **Реализация**: используется `Send` (или параллельные ветки) для запуска нескольких экземпляров одного узла или разных узлов одновременно. Агрегатор ждёт завершения всех и объединяет результаты.
- **Примечание**: параллелизм возможен только для узлов, не имеющих зависимостей по данным.


#### Routing <a id="routing"></a>
- **Суть**: входные данные попадают в router node, который анализирует их и направляет в соответствующий worker node (или ветку).
- **Реализация**: `conditional_edge` вызывает функцию классификации, возвращающую имя следующего узла.


#### Orchestrator-Worker <a id="orchestrator-worker"></a>
- **Суть**: центральный оркестратор анализирует задачу, динамически определяет список подзадач и делегирует их воркерам, а затем собирает и синтезирует финальный ответ.
- **Отличие от Parallelization**: оркестратор сам решает, какие воркеры создать (часто с разными промптами/инструментами). Может быть реализован через динамические параллельные ветки (например, `Send` внутри узла-оркестратора).


#### Evaluator-Optimizer <a id="evaluator-optimizer"></a>
- **Суть**: генератор создаёт ответ, а evaluator оценивает его по заданным критериям. Если ответ не проходит, evaluator возвращает обратную связь (feedback) генератору для улучшения. Цикл повторяется до достижения качества.
- **Реализация**: цикл с conditional edges: после evaluator либо переход к генератору (с feedback в State), либо выход.

#### Определение State

#### Вариант 1: Классический TypedDict с редуктором

```python
class StateTypedDict(TypedDict):
    messages: Annotated[list, add_messages]
    counter: int
    user_id: str
```

#### Вариант 2: Использование встроенного MessagesState (удобно для чата)

```python
class StateMessages(MessagesState):
    # автоматически есть поле messages с add_messages
    counter: int
    user_id: str
```

#### Вариант 3: StateSchema (новое в 1.0) — с ReducedValue, MessagesValue, UntrackedValue

```python
from langgraph.graph import StateSchema, ReducedValue, MessagesValue
from typing import Annotated
import operator

class StateSchemaV1(StateSchema):
    # MessagesValue — встроенный тип для сообщений
    messages: MessagesValue
    # Обычное поле
    counter: int
    # Поле с кастомным редуктором (накопление списка)
    results: Annotated[List[str], operator.add]
    # Поле, которое НЕ будет сохраняться в чекпоинтах (runtime данные)
    temp_cache: UntrackedValue[dict]
```

#### Узлы (Nodes)

```python
# Простая функция узла
def node_a(state: StateTypedDict) -> dict:
    return {"counter": state.get("counter", 0) + 1}

# Node с дополнительными аргументами (store, config, writer)
def node_with_store(state: StateTypedDict, store: InMemoryStore, config: dict, writer: StreamWriter):
    # writer можно использовать для кастомного стриминга
    writer({"custom": "data"})
    # читаем из store
    user_id = config["configurable"]["user_id"]
    memories = store.search((user_id, "preferences"))
    return {"messages": [AIMessage(content=f"Found {len(memories)} memories")]}

# Асинхронный Node
async def async_node(state: StateTypedDict):
    # имитация асинхронного вызова
    return {"counter": 42}
```

#### Edges и Conditional Edges

```python
# Fixed Edges: после node_a идём в node_b
# builder.add_edge("node_a", "node_b")

# Conditional Edge: функция возвращает имя следующего узла
def router(state: StateTypedDict) -> Literal["node_b", "node_c"]:
    if state["counter"] > 5:
        return "node_b"
    return "node_c"

# Можно использовать в builder.add_conditional_edges("node_a", router)
```

#### Построение простого графа

```python
builder = StateGraph(StateTypedDict)
builder.add_node("a", node_a)
builder.add_node("b", lambda s: {"messages": [AIMessage(content="Done")]})
builder.add_edge(START, "a")
builder.add_conditional_edges("a", router)
builder.add_edge("b", END)

# Компиляция с чекпоинтером
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Запуск
config = {"configurable": {"thread_id": "thread1", "user_id": "user123"}}
result = graph.invoke({"counter": 0, "messages": []}, config)
print(result)
```

#### Быстрый старт с StateSchema (новое в 1.0)

```python
# Полный пример с новым синтаксисом
class MyState(StateSchema):
    messages: MessagesValue
    counter: int
    temp: UntrackedValue[dict]

builder = StateGraph(MyState)
builder.add_node("inc", lambda s: {"counter": s["counter"] + 1})
builder.add_edge(START, "inc")
builder.add_edge("inc", END)
graph = builder.compile(checkpointer=MemorySaver())

state = graph.invoke({"messages": [], "counter": 0}, config={"configurable": {"thread_id": "test"}})
print(state)
```

---

## Module 2: Управление State (Persistence) <a id="module-2"></a>

LangGraph предоставляет встроенный механизм контрольных точек (checkpointing) для сохранения State графа между запусками и возможности воспроизведения (replay) или отладки.

### 1. Основные понятия <a id="basic"></a>

- **Checkpointer** – объект, который сохраняет State графа после каждого супер-шага (super-step). Включает в себя хранилище (например, в памяти, на диске или в БД) и логику сериализации.

- **Thread (поток)** – логический контейнер для серии чекпоинтов, принадлежащих одному разговору/сессии. Идентифицируется `thread_id` в конфигурации.

- **Super-step** – атомарная единица выполнения графа. Это **не синтаксический элемент** (node), а **единица исполнения**, в рамках которой выполняются все узлы, готовые к запуску (у которых выполнены зависимости по данным). После завершения супер-шага State детерминировано и сохраняется в чекпоинт.

  - Если node A изменяет State, и node B должно видеть это изменение, то они выполняются в **разных** супер-шагах (сначала A, затем B).

  - Если узлы независимы и могут выполняться параллельно, они могут быть в **одном** супер-шаге (если их запуск запланирован одновременно).

- **Checkpoint** – снимок State графа в конце супер-шага. Содержит:

  - `state` – актуальное State.

  - `metadata` – временные метки, номер супер-шага, идентификатор чекпоинта, следующий node (или Nodes), запланированные к исполнению.

- **StateSnapshot** – удобное представление чекпоинта для чтения (возвращается методами `get_state` / `get_state_history`).

### 2. Работа с чекпоинтами <a id="checkpoints-work"></a>

- Сохранение происходит автоматически при использовании checkpointer'а в графе.
- Конфигурация запуска всегда содержит `thread_id`:
- Каждый `thread_id` хранит свою собственную историю чекпоинтов. Это позволяет графу «помнить» предыдущие взаимодействия в рамках одного диалога.
- При каждом новом запуске с тем же `thread_id` граф автоматически восстанавливает последнее State и продолжает с него (если не указан конкретный `checkpoint_id`).
- Таким образом, можно строить долгоживущие диалоговые агенты с памятью.

  ```python
  config = {"configurable": {"thread_id": "user_123_conversation_1"}}
  ```

- **Получение последнего State**:

  ```python
  snapshot = graph.get_state(config)

  ```

- **История State** (все чекпоинты потока):

  ```python
  history = graph.get_state_history(config)
  # Возвращает список StateSnapshot, начиная с самого свежего.
  ```


### 3. Воспроизведение (Replay) <a id="replay"></a>

- Позволяет «перемотать» выполнение к любому чекпоинту и продолжить с того места.

- Для этого нужно передать в `invoke` конфигурацию, содержащую **как thread_id, так и checkpoint_id** нужного чекпоинта.

  ```python
  replay_config = {
      "configurable": {
          "thread_id": "user_123_conversation_1",
          "checkpoint_id": "1ef3a7b...",  # ID из истории
      } }
  graph.invoke(None, replay_config)  # None означает "продолжить с сохранённого State"
  ```

- Выполнение начнётся с того узла (или узлов), который был запланирован после восстановленного чекпоинта (поле `next` в StateSnapshot).

### 4. Ручное обновление State (update_state) <a id="update-state"></a>


- Позволяет модифицировать State в существующем чекпоинте **без выполнения узлов**.
- Все операции (`get_state`, `update_state`, `invoke` с replay) работают в рамках одного checkpointer'а.
- Полезно для отладки, исправления ошибок, «подсказок» графу.

- **Базовый вариант**:

  ```python
  graph.update_state(config, {"messages": [{"role": "user", "content": "new input"}]})

  ```

  Обновит State в последнем чекпоинте потока. Указатель выполнения (`next`) **не двигается**.

- **Параметр `as_node`**:

  ```python
  graph.update_state(
      config,
      {"messages": [{"role": "assistant", "content": "corrected answer"}]},
      as_node="assistant_node" )
  ```
  - Это способ сказать LangGraph: «считай, что node `assistant_node` выполнился и вернул такое обновление State.

  - При этом:

    - Сам Node **не вызывается**.

    - Редукторы (reducers) **не применяются** – вы должны передать полное значение, которое ожидается после выполнения узла (как если бы Node вернул его).

    - Указатель выполнения (`next`) **перемещается** так, будто Node действительно выполнился (т.е. следующими будут узлы, идущие после `assistant_node` по графу).

  - Это особенно полезно для ручной коррекции в графах с циклами или условными переходами: вы можете «подменить» результат узла и продолжить выполнение с правильным State.

### 5. Ручное обновление State (as_node) <a id="as_node"></a>

- `update_state` без `as_node` просто меняет данные, но не трогает `next`. Это значит, что если после этого вызвать `invoke`, выполнение продолжится с того же места (узла), который был запланирован до обновления.

- `update_state` с `as_node` позволяет «вставить» результат выполнения узла задним числом, что может изменить дальнейший маршрут графа.

- При использовании `as_node` важно точно указать имя узла, которое существует в графе. Иначе возникнет ошибка.


#### Persistence (Checkpoints) — работа с чекпоинтами

```python
# После запуска графа с чекпоинтером

# Получить последний State
snapshot = graph.get_state(config)
print(snapshot.values)

# Получить историю всех чекпоинтов
history = list(graph.get_state_history(config))
print(f"Всего чекпоинтов: {len(history)}")
print(f"Последний checkpoint_id: {history[0].config['configurable']['checkpoint_id']}")

# Переместиться к конкретному чекпоинту (replay)
old_checkpoint_id = history[-1].config["configurable"]["checkpoint_id"]  # самый первый
replay_config = {"configurable": {"thread_id": "thread1", "checkpoint_id": old_checkpoint_id}}
graph.invoke(None, replay_config)  # продолжит с того места

# Ручное обновление State (без выполнения узлов)
graph.update_state(config, {"counter": 100})
# Теперь State изменилося, но next не двигался

# Обновление с as_node — "имитируем" выполнение узла
graph.update_state(config, {"messages": [AIMessage(content="Corrected")]}, as_node="b")
# Теперь указатель next переместится так, будто node b выполнился
```

---

## Module 3: Продвинутое управление State и устойчивость выполнения (Durability)

### 1. Checkpoint и CheckpointInterface <a id="module3-checkpoint-interface"></a>

- **`BaseCheckpointSaver`** – это абстрактный базовый класс (интерфейс), который определяет, как чекпоинтер должен сохранять и загружать чекпоинты. Все конкретные реализации чекпоинтеров (например, `MemorySaver`, `SqliteSaver`, `PostgresSaver`) наследуются от него.

- **CheckpointInterface** – это набор методов, которые обязан реализовать любой чекпоинтер:

  - `put` – сохранить чекпоинт для данного треда и супер-шага.

  - `put_writes` – сохранить промежуточные записи (pending writes), которые ещё не оформлены в полноценный чекпоинт (см. ниже).

  - `get` – получить последний чекпоинт треда.

  - `get_tuple` – получить конкретный чекпоинт по его идентификатору.

  - `list` – получить историю чекпоинтов треда.

- Это позволяет легко подключать различные хранилища (память, файлы, базы данных) без изменения логики графа.


### 2. Ключевые возможности на базе чекпоинтов <a id="module3-key-features"></a>

#### Human-in-the-loop (HITL) and Interrupts <a id="module3-hitl"></a>

- Реализуется с помощью **прерываний (interrupts)**. Вы можете установить точку останова перед определённым узлом, используя `NodeInterrupt` или параметр `interrupt_before` при компиляции графа.
- LangGraph хорошо интегрируется с инструментами (tools) и поддерживает паузы для взаимодействия с человеком.

- При достижении прерывания выполнение останавливается, State сохраняется в чекпоинте, и управление возвращается вызывающему коду. Затем можно:
  - Запросить ввод пользователя.
  - Вручную скорректировать State через `update_state`.
  - Дать команду на продолжение (`.invoke` с тем же `thread_id`).
- Это основа для сценариев, где требуется одобрение человека или ввод дополнительных данных.

Interrupts позволяют остановить выполнение графа в произвольной точке, дождаться внешнего воздействия (например, ввода пользователя) и затем возобновить. Это ключевой механизм для реализации Human-in-the-loop (HITL). 
- Можно установить точку останова перед определённым узлом. Выполнение остановится, State сохранится, и можно будет вмешаться вручную (через update_state или просто запросить подтверждение).


#### Базовый механизм

Внутри узла можно вызвать `interrupt(value)`, где `value` — произвольные JSON-сериализуемые данные, которые будут переданы вызывающему коду.


```python
from langgraph.types import interrupt

def human_review_node(state):
    # запрашиваем подтверждение у пользователя
    user_input = interrupt("Нужно подтверждение перед вызовом API")

    # после возобновления user_input будет содержать данные, переданные через Command(resume=...)
    return {"feedback": user_input}

```

При достижении `interrupt` выполнение графа приостанавливается, и управление возвращается коду, который вызвал граф. При этом:

- создаётся чекпоинт с сохранённым State;

- вызывающий получает объект `Interrupt` с переданным значением и информацией о том, где произошло прерывание.

#### Возобновление после прерывания

Чтобы продолжить, нужно вызвать граф с тем же `thread_id` и передать `Command(resume=...)`:


```python
from langgraph.types import Command

# после получения прерывания
command = Command(resume={"approved": True, "comment": "Ок"})
graph.invoke(command, config={"configurable": {"thread_id": "..."}})

```

Значение, переданное в `resume`, станет возвращаемым значением вызова `interrupt` внутри узла. Таким образом, код после `interrupt` получает внешние данные.

**Важно:** при возобновлении весь код узла **выполняется заново с начала** (до `interrupt`), поэтому:

- операции до `interrupt` должны быть идемпотентными;

- нежелательные side-effects (например, вызов внешнего API) лучше перенести после `interrupt` или в отдельный Node.


#### Правила использования Interrupts

1. **Не оборачивайте `interrupt` в `try/except`**, иначе исключение не дойдёт до графа и прерывание не сработает корректно.

2. **Порядок вызовов `interrupt` в узле имеет значение**. Если в узле несколько прерываний, LangGraph создаёт список значений для возобновления и сопоставляет их по индексу. Менять местами или пропускать прерывания нельзя.

3. **Избегайте условного пропуска interrupt** – это нарушит сопоставление.

4. **Возвращайте из `interrupt` только простые JSON-сериализуемые типы** (dict, list, str, int, float, bool, None).

5. **Идемпотентность**: перед interrupt ставьте только операции, которые можно безопасно повторить (чтение данных, локальные вычисления). Все необратимые действия (запись в БД, отправка email) – после interrupt или в отдельном узле, который запускается после прерывания.


#### Распространённые паттерны

- **Подтверждение (Approval workflows)** – пауза перед критическими операциями (вызов API, изменение БД).

- **Множественные прерывания** – например, запрос уточняющих данных на разных этапах. Важно соблюдать порядок.

- **Редактирование и ревью** – прерывание после генерации текста для правки пользователем.

- **Прерывание во время вызова инструментов** – если инструмент требует подтверждения, можно вызвать interrupt внутри обработчика инструмента.


#### Timetravel (путешествие во времени) <a id="module3-timetravel"></a>

- Благодаря сохранению всей истории чекпоинтов вы можете «отмотать» выполнение к любому предыдущему моменту и запустить альтернативную ветку.
- Для этого используется **replay** (воспроизведение) с указанием нужного `checkpoint_id`. Затем можно применить `update_state` с `as_node`, чтобы скорректировать решение, и продолжить выполнение уже по новому пути.
- Это мощный инструмент для отладки, экспериментов и исправления ошибок в уже выполненных сценариях.

#### Основные шаги

1. **Запустить граф с чекпоинтером**, чтобы сохранялась история State.

2. **Получить историю чекпоинтов** для нужного треда:

   ```python
   history = graph.get_state_history(config)
   # history — список StateSnapshot, последний (самый свежий) идёт первым
   ```

3. **Выбрать нужный чекпоинт** (например, предпоследний) и получить его `checkpoint_id`:

   ```python
   target_snapshot = history[3]  # например
   checkpoint_id = target_snapshot.config["configurable"]["checkpoint_id"]
   ```

4. **Перезапустить граф с этого чекпоинта** (replay):

   ```python
   replay_config = {"configurable": {"thread_id": "...", "checkpoint_id": checkpoint_id}}
   graph.invoke(None, replay_config)  # None означает "продолжить с сохранённого State"
   ```

   Выполнение начнётся с узла (или узлов), запланированных после этого чекпоинта.

5. **(Опционально) изменить State перед продолжением** через `update_state`:

   ```python
   graph.update_state(replay_config, {"user_input": "другое значение"}, as_node="some_node")
   # после этого можно снова invoke
   ```

#### Применения Time-Travel

- **Анализ решений**: понять, почему граф пришёл к определённому результату.

- **Отладка ошибок**: воспроизвести проблемный момент и исследовать State.

- **Исследование альтернатив**: изменить одно решение и посмотреть, как изменится итог.


#### Отказоустойчивость (Fault-tolerance) и ожидающие записи (Pending writes) <a id="module3-fault-tolerance"></a>

- **Fault-tolerance**: если во время выполнения узла произошёл сбой (например, падение процесса), при следующем запуске с тем же `thread_id` граф восстановится из последнего сохранённого чекпоинта. Однако нужно учитывать, что операции внутри узла могли быть выполнены частично. Для обеспечения идемпотентности и избежания повторного выполнения небезопасных операций используется механизм **pending writes**.

- **Pending writes** – это временные записи в State, которые ещё не зафиксированы в чекпоинте. Они возникают, когда Node возвращает обновление State, но супер-шаг ещё не завершён (например, если Node работает в фоне или были вызваны инструменты). Чекпоинтер может сохранять эти pending writes отдельно, чтобы при восстановлении после сбоя не потерять промежуточные результаты. В LangGraph pending writes автоматически управляются чекпоинтером, гарантируя, что State всегда будет консистентным после завершения супер-шага.


### 3. Durable Execution (Устойчивое выполнение) <a id="module3-durable-execution"></a>


**Durable Execution** – это техника, позволяющая процессу (графу) быть устойчивым к сбоям и остановкам: выполнение может быть прервано в любой момент, а затем возобновлено с того же места без потери прогресса.


#### Как это работает в LangGraph <a id="module3-how-it-works"></a>

- Основа – **чекпоинты**, которые сохраняют State после каждого super-step.

- Если процесс прерывается (сбой, пауза для HITL, добровольная остановка), то при следующем запуске с тем же `thread_id` граф загружает последний чекпоинт и продолжает с того узла (узлов), которые были запланированы (`next`).

- Однако важно, чтобы **недетерминированные операции** (вызовы внешних API, генерация случайных чисел, работа с реальным временем) не повторялись при возобновлении, иначе можно получить некорректные результаты или двойные списания. Для этого используется декоратор `@task`.


#### Обёртывание недетерминированных операций в `@task` <a id="module3-task"></a>

- `@task` – это декоратор из LangGraph (`langgraph.task`), который превращает обычную функцию в **задачу** (task) с поддержкой durable execution.

- Когда вы вызываете такую задачу внутри узла, её результат автоматически сохраняется в чекпоинте. При повторном запуске узла (после восстановления) задача **не выполняется заново**, а возвращает ранее сохранённый результат.

- Это позволяет безопасно выполнять операции с сайд-эффектами: они гарантированно выполняются только один раз в рамках данного экземпляра выполнения.

- **Пример**:

  ```python
  from langgraph.task import task

  @task
  def call_llm(prompt: str) -> str:
      # здесь может быть вызов OpenAI API
      return actual_llm_call(prompt)

  def my_node(state: State):
      # результат будет закеширован в чекпоинте
      response = call_llm(state["prompt"])
      return {"response": response}

  ```

#### Важный нюанс при возобновлении <a id="module3-resume-nuance"></a>

- Если произошла остановка (сбой или пауза), возобновление начинается **не с точной строки кода, где остановился**, а с **начала того супер-шага**, который не был завершён. Это означает, что все операции внутри узла, не обёрнутые в `@task`, будут выполнены заново. Поэтому рекомендуется оборачивать в `@task` все функции, которые не должны повторяться.

- При использовании `@task` вы получаете прозрачное восстановление: при повторном входе в node задача вернёт уже вычисленное значение.

### 4. Durability Modes (режимы сохранения) <a id="mmodule3-durability-modes3"></a>


В официальной документации LangGraph нет встроенного параметра `durability` в методах `stream` или `invoke`. Однако концептуально можно выделить разные стратегии сохранения State, которые зависят от выбранного чекпоинтера и его настроек:


- **exit** (сохранение только при выходе) – State сохраняется только после полного завершения графа (или при явном прерывании). Это поведение соответствует чекпоинтеру, который пишет чекпоинты только при завершении работы. В LangGraph по умолчанию чекпоинты сохраняются **после каждого супер-шага**, что ближе к режиму `sync`.

- **async** (асинхронное сохранение) – чекпоинты пишутся асинхронно, не блокируя выполнение графа. Это может быть реализовано, например, с помощью чекпоинтера, который ставит запись в очередь и подтверждает сохранение позже. В стандартных чекпоинтерах (`MemorySaver`, `SqliteSaver`) запись обычно синхронная, но можно создать свою реализацию `BaseCheckpointSaver` с асинхронным `put`.

- **sync** (синхронное сохранение) – каждый чекпоинт записывается синхронно, гарантируя, что State сохранено до перехода к следующему супер-шагу. Это поведение по умолчанию для встроенных чекпоинтеров.


На практике выбор режима влияет на производительность и гарантии durability. Для большинства сценариев достаточно синхронной записи после каждого супер-шага.


### 5. Использование Task в узлах (более подробно) <a id="module3-task-details"></a>



- **Зачем нужны `@task`?**

  - Чтобы избежать повторного выполнения кода при восстановлении после сбоя или прерывания.

  - Чтобы управлять долгими операциями (например, вызов LLM) и иметь возможность их прервать и возобновить.

- **Как это работает?**

  - При первом вызове задачи её результат сохраняется в чекпоинте (в специальном разделе, связанном с текущим супер-шагом).

  - При повторном вызове (в том же или в восстановленном выполнении) задача не запускается, а сразу возвращает сохранённое значение.

  - Задачи могут принимать аргументы и возвращать результаты, которые сериализуются.

- **Ограничения:**

  - Аргументы и результат должны быть сериализуемы (JSON-совместимы или через pickle, в зависимости от чекпоинтера).

  - Задачи не должны иметь побочных эффектов, не сохраняемых в State (например, запись в файл без синхронизации), иначе при восстановлении эффект может быть утерян или продублирован.

- **Альтернатива**: можно использовать `@task` для вызова инструментов, чтобы гарантировать идемпотентность.



### 6. Возобновление workflows (Resuming Workflows) <a id="module3-resuming"></a>

#### Пауза и возобновление через Interrupts и Commands <a id="module3-interrupts-commands"></a>

- **Interrupts** – механизм принудительной остановки графа. Может быть вызван явно из узла с помощью `NodeInterrupt` или установлен при компиляции (`interrupt_before`).

- После остановки State сохраняется, и управление возвращается. Чтобы продолжить, нужно вызвать `graph.invoke` или `graph.stream` с тем же `thread_id` (и, опционально, с `Command`). `Command` позволяет передать данные или указать, какой Node выполнить следующим.

- **Пример**:

  ```python

  # после прерывания
  graph.invoke(None, config={"configurable": {"thread_id": "..."}})

  # или с командой
  graph.invoke(Command(resume="user_approval"), config)

  ```

#### Восстановление после ошибок через указание checkpoint_id <a id="module3-error-recovery"></a>

- Если выполнение упало с исключением, последний чекпоинт сохранился (если чекпоинтер используется). Можно проанализировать State и, если нужно, откатиться к предыдущему чекпоинту.

- Чтобы возобновить с более ранней точки, используйте replay:

  ```python

  # получить нужный checkpoint_id из истории
  snapshot = graph.get_state_history(config)[n]  # например, предпоследний
  replay_config = {"configurable": {"thread_id": "...", "checkpoint_id": snapshot.config["configurable"]["checkpoint_id"]}}
  graph.invoke(None, replay_config)
  ```

- После этого можно применить `update_state` для коррекции и продолжить выполнение.


#### Durable Execution — декоратор @task

```python
@task
def fetch_weather(city: str) -> str:
    # Этот вызов будет выполнен только один раз в рамках одного супер-шага,
    # даже при восстановлении после сбоя
    # реальный вызов API
    return f"Sunny in {city}"

def node_with_task(state: StateTypedDict):
    weather = fetch_weather("Moscow")
    return {"messages": [AIMessage(content=weather)]}
```

#### 10. Interrupts (Human-in-the-Loop)

```python
def node_with_interrupt(state: StateTypedDict):
    # Важная операция, требующая подтверждения
    user_input = interrupt("Подтвердите действие Y/N")
    if user_input == "Y":
        # выполняем действие
        return {"result": "approved"}
    else:
        return {"result": "rejected"}

# В коде вызова графа:
try:
    graph.invoke({"input": "..."}, config)
except Exception as e:  # На практике ловим Interrupt
    print("Прерывание:", e)
    # Принимаем решение и продолжаем
    graph.invoke(Command(resume="Y"), config)
```

#### Правила interrupt: порядок и идемпотентность

```python
# Несколько interrupt в одном узле — важен порядок
def node_multi_interrupt(state):
    x = interrupt("step1")
    y = interrupt("step2")  # при возобновлении сначала получит значение для первого interrupt, потом для второго
    return {"x": x, "y": y}

# При возобновлении нужно передать список значений:
# Command(resume=["value1", "value2"])
```

#### Time Travel

```python
# Получаем историю
history = list(graph.get_state_history(config))
# Берём, например, 3-й чекпоинт
target = history[2]
replay_config = target.config  # содержит thread_id и checkpoint_id

# Запускаем с этого места (replay)
graph.invoke(None, replay_config)

# Можно изменить State перед продолжением
graph.update_state(replay_config, {"counter": 999})
graph.invoke(None, replay_config)  # продолжит с изменённым State
```

---

## Module 4: Управление памятью в LangGraph 



LangGraph предоставляет два уровня памяти:

1. **Short-term memory (кратковременная)** – хранится в чекпоинтах и привязана к конкретному потоку (thread). Используется для поддержания контекста диалога внутри одной сессии.

2. **Long-term memory (долговременная)** – хранится в отдельном хранилище `Store` и доступна между разными тредами и сессиями. Позволяет сохранять информацию о пользователе, его предпочтениях и т.д.



### 1. Store – абстракция долговременной памяти <a id="module4-store"></a>



**`BaseStore`** – это интерфейс для хранилища, который определяет методы работы с данными: `put`, `get`, `search`, `delete`. Конкретные реализации:

- `InMemoryStore` – хранит данные в памяти (для тестирования и разработки).

- `PostgresStore`, `RedisStore` – для production (требуют настройки БД).



#### Основные понятия Store: <a id="module4-store-concepts"></a>

- **Namespace** – кортеж строк (`tuple[str, ...]`), задающий пространство имён для группировки записей. Обычно первым элементом указывается идентификатор пользователя, затем категория памяти.

- **Key** – уникальный строковый идентификатор записи в рамках namespace (можно генерировать через `uuid`).

- **Value** – произвольный JSON-сериализуемый словарь с данными.



#### Пример работы с InMemoryStore: <a id="module4-store-example"></a>

```python

import uuid
from langgraph.store.memory import InMemoryStore

# 1. Создаём store
store = InMemoryStore()

# 2. Определяем namespace (например, для пользователя с id = "user123")
user_id = "user123"
namespace = (user_id, "memories")  # кортеж строк

# 3. Генерируем ключ для новой записи
key = str(uuid.uuid4())

# 4. Данные для сохранения
value = {"food_preference": "I like pizza", "last_updated": "2025-02-27"}

# 5. Сохраняем
store.put(namespace, key, value)

```

#### Поиск и получение данных: <a id="mmodule4-store-search3"></a>

```python

# Получить все записи в namespace (возвращает список объектов Item)
items = store.search(namespace)
for item in items:
    print(item.key, item.value)

# Получить конкретную запись по namespace и key
retrieved = store.get(namespace, key)
print(retrieved.value)  # {"food_preference": "I like pizza", ...}

```


#### Семантический поиск (Semantic Search) <a id="module4-semantic-search"></a>

- Для поиска не по ключам, а по смыслу, нужен store с поддержкой векторов. В текущей версии LangGraph (0.2.x) есть `InMemoryStore`, но он не выполняет семантический поиск «из коробки».

- Можно создать свою реализацию `BaseStore`, которая использует векторизацию (например, через `sentence-transformers`) и хранит эмбеддинги. Или использовать внешние векторные БД (Pinecone, Weaviate, pgvector) и реализовать методы `put` и `search` с учётом эмбеддингов.

- Пример упрощённого подхода: хранить эмбеддинги вместе с value и при поиске вычислять косинусное сходство с запросом. Но для серьёзных проектов лучше интегрировать специализированное хранилище.

#### Добавление Store в граф <a id="module4-store-in-graph"></a>
 
```python

builder = StateGraph(State)
# ... добавляем Nodes and Edges ...
graph = builder.compile(store=store)  # передаём store при компиляции

```
Теперь внутри узлов графа можно обращаться к хранилищу через аргумент `store` (если он указан в функции узла). Например:

```python

def my_node(state, store):
    # Получаем данные пользователя из store
    user_id = state["user_id"]
    memories = store.search((user_id, "memories"))
    # ... используем memories ...
    return {"some": "output"}

```

### 2. Short-term memory (кратковременная память) <a id="module4-short-term"></a>



- Реализуется через **чекпоинты (checkpointers)**. После каждого super-step State графа автоматически сохраняется в чекпоинт.

- Чекпоинт хранит всё State, включая историю сообщений, переменные и т.д.

- Доступные чекпоинтеры:

  - `MemorySaver` – в памяти (для разработки).

  - `PostgresSaver` – сохранение в PostgreSQL.

  - `MongoDBSaver` – в MongoDB.

  - `RedisSaver` – в Redis.

  - Есть также `SqliteSaver` для SQLite.

- Привязка к `thread_id` обеспечивает изоляцию между разными диалогами.


**Пример использования кратковременной памяти:**

```python

from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Первый запуск с thread_id="conv_1"
config = {"configurable": {"thread_id": "conv_1"}}
graph.invoke({"messages": []}, config)

# Второй запуск с тем же thread_id продолжит разговор
graph.invoke({"messages": [HumanMessage(content="Привет!")]}, config)

```

### 3. Long-term memory vs Short-term memory <a id="module4-comparison"></a>

| Характеристика | Short-term (чекпоинты) | Long-term (Store) |
|----------------|-------------------------|-------------------|
| Область видимости | В рамках одного треда (`thread_id`) | Глобально (между тредами, пользователями) |
| Цель | Поддержание контекста текущего диалога | Сохранение знаний о пользователе, предпочтений, фактов |
| Автоматическое сохранение | Да, после каждого супер-шага | Нет, только явные операции `put` |
| Типичные хранилища | `MemorySaver`, `PostgresSaver`, `RedisSaver` | `InMemoryStore`, `PostgresStore`, `RedisStore` |
| Поиск | Только по чекпоинтам (ключи – время/супер-шаг) | По ключам (namespace + key) + опционально семантический |

### 4. Управление длиной контекста (Context Management) <a id="module4-context-management"></a>


При продолжительном диалоге история сообщений может превысить лимит контекстного окна LLM. LangGraph предлагает несколько стратегий работы с этим:


#### 1. Trim (обрезание) <a id="module4-trim"></a>

- Удалить самые старые или самые новые сообщения, чтобы оставить только последние N.

- Можно реализовать в отдельном узле, который перед вызовом LLM обрабатывает список сообщений.

```python

from langchain_core.messages import trim_messages

def trimmer_node(state):
    trimmed = trim_messages(
        state["messages"],
        max_tokens=2000,  # лимит токенов
        strategy="last",  # сохранить последние сообщения
        token_counter=...  # функция подсчёта токенов
    )
    return {"messages": trimmed}

```

#### 2. Delete (удаление) <a id="module4-delete"></a>

- Полное удаление определённых сообщений или всей истории.

- В LangGraph можно манипулировать State напрямую (например, `state["messages"] = state["messages"][-10:]`).



#### 3. Summarize (суммаризация) <a id="module4-summarize"></a>

- Использовать LLM для сжатия истории в краткое резюме, а затем хранить это резюме вместо части сообщений.

- Например, можно суммаризировать каждые K сообщений и сохранять резюме как системное сообщение.

```python

def summarize_node(state):
    if len(state["messages"]) > 20:
        summary_prompt = "Суммаризируй следующий диалог: " + "\n".join([m.content for m in state["messages"][:-10]])
        summary = llm.invoke(summary_prompt)  # получаем резюме
        # заменяем старые сообщения на одно резюме
        new_messages = [SystemMessage(content=f"Краткое содержание предыдущего разговора: {summary}")] + state["messages"][-10:]
        return {"messages": new_messages}
    return {}

```

#### 4. Manage Checkpoints (управление чекпоинтами) <a id="module4-manage-checkpoints"></a>

- Можно использовать историю чекпоинтов (`graph.get_state_history`) для доступа к предыдущим State и, например, извлекать оттуда суммаризации или важные факты, не храня все сообщения в текущем State.

- Это требует ручного управления: при каждом шаге вы решаете, какие данные из прошлого вам нужны, и подгружаете их.


### 5. Миграции баз данных <a id="module4-migrations"></a>


При использовании production-хранилищ (PostgresStore, PostgresSaver и т.д.) необходимо заранее создать соответствующие таблицы. LangGraph предоставляет вспомогательные методы:


```python

from langgraph.checkpoint.postgres import PostgresSaver

# Создание таблиц для чекпоинтов (обычно выполняется однократно)
with connection as conn:
    saver = PostgresSaver(conn)
    saver.setup()  # создаёт таблицы, если их нет

```

**Рекомендация:** выносить создание таблиц в отдельный скрипт инициализации или миграции (например, с использованием Alembic для SQLAlchemy). Это гарантирует, что перед запуском приложения схема БД будет корректной.

Для Store (например, `PostgresStore`) также есть метод `setup()`.




#### Long-term Memory — Store

```python
# Создаём in-memory store
store = InMemoryStore()

# Сохраняем память о пользователе
namespace = ("user123", "preferences")
key = str(uuid.uuid4())
value = {"theme": "dark", "language": "ru"}
store.put(namespace, key, value)

# Ищем все записи в namespace
items = store.search(namespace)
for item in items:
    print(item.key, item.value)

# Получаем конкретную запись
retrieved = store.get(namespace, key)
print(retrieved.value)

# Подключаем store к графу
graph_with_store = builder.compile(checkpointer=checkpointer, store=store)

# В узле можно получить store через аргумент (как в node_with_store выше)
```

#### Семантический поиск (имитация)

```python
# Для настоящего семантического поиска нужна векторная БД.
# Пример с использованием эмбеддингов (упрощённо):
from sentence_transformers import SentenceTransformer

class VectorStore(InMemoryStore):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        super().__init__()
        self.model = SentenceTransformer(model_name)
        self.embeddings = {}
    
    def put(self, namespace, key, value):
        super().put(namespace, key, value)
        # сохраняем эмбеддинг для текстового поля
        text = value.get("text", "")
        self.embeddings[(namespace, key)] = self.model.encode(text)
    
    def search_similar(self, namespace, query, top_k=5):
        query_emb = self.model.encode(query)
        scores = []
        for (ns, key), emb in self.embeddings.items():
            if ns == namespace:
                score = cosine_similarity([query_emb], [emb])[0][0]
                scores.append((key, score))
        scores.sort(key=lambda x: -x[1])
        return [self.get(namespace, key) for key, _ in scores[:top_k]]
```

---

## Module 5: Потоковая передача и подграфы (Streaming & Subgraphs)

### 1. Streaming (потоковая передача) <a id="streaming"></a>

Streaming позволяет получать данные от графа в реальном времени по мере выполнения. Это полезно для:
- отображения частичных результатов пользователю (например, токенов LLM по мере генерации);
- Поддержка потоковой передачи токенов, событий и обновлений State в реальном времени.
- мониторинга прогресса;
- отладки.


#### Режимы стриминга (`stream_mode`) <a id="streaming-modes"></a>

При вызове `graph.stream(input, stream_mode=...)` можно указать один или несколько режимов:


| Режим        | Описание |

|--------------|----------|

| `"values"`   | Передаёт **полный State** после каждого супер-шага. |

| `"updates"`  | Передаёт только **обновления**, возвращённые узлами (т.е. то, что каждый Node вернул как изменение State). |

| `"messages"` | Специализированный режим для чат-моделей: передаёт сообщения по мере генерации (токены). Требует, чтобы узлы возвращали сообщения в формате LangChain. |

| `"custom"`   | Позволяет узлам отправлять произвольные данные через `write_custom_data` (продвинутое использование). |

| `"debug"`    | Детальная информация о каждом шаге выполнения (для отладки). |


Можно комбинировать режимы, передавая список: `stream_mode=["updates", "messages"]`.

**Пример использования:**

```python

async for chunk in graph.astream(
    {"messages": [HumanMessage(content="Напиши короткий рассказ")]},
    stream_mode=["updates", "messages"]
):

    if "messages" in chunk:
        # это токены сообщения
        print(chunk["messages"], end="")
    elif "updates" in chunk:
        # обновления State от узлов
        print(f"Обновление: {chunk['updates']}")

```

**Важно:** режим `"messages"` работает только если узлы возвращают сообщения с поддержкой стриминга (например, через `.astream()` вызов LLM). Подробнее в документации LangChain.

### 4. Subgraph (подграфы) <a id="subgraph"></a>

Subgraph позволяет инкапсулировать часть логики в отдельный граф со своим State, а затем использовать его как Node в родительском графе. Это способствует модульности, повторному использованию и разделению ответственности.


#### Как это работает <a id="subgraph-how"></a>


- Subgraph — это полностью самостоятельный граф, скомпилированный отдельно.

- Родительский граф включает subgraph как Node с помощью метода `add_node`, передавая сам subgraph.

- **State**: между родительским и дочерним графами происходит синхронизация по **пересекающимся ключам (overlapping keys)**. Если в определении State родительского и дочернего графа есть одинаковые ключи, их значения будут автоматически передаваться в subgraph при входе и обновляться при выходе.



#### Пример <a id="subgraph-example"></a>


```python
from langgraph.graph import StateGraph, MessagesState
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

# State подграфа
class SubgraphState(TypedDict):
    messages: Annotated[list, add_messages]
    sub_custom: str

# Создаём подграф
subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node("sub_node", lambda state: {"sub_custom": "обработано"})
subgraph_builder.set_entry_point("sub_node")
subgraph_builder.set_finish_point("sub_node")
subgraph = subgraph_builder.compile()

# Родительский State
class ParentState(TypedDict):
    messages: Annotated[list, add_messages]  # общий ключ
    parent_custom: str

# Родительский граф
builder = StateGraph(ParentState)
builder.add_node("subgraph", subgraph)  # добавляем подграф как Node
builder.add_node("other", ...)
builder.set_entry_point("subgraph")
builder.add_edge("subgraph", "other")
parent_graph = builder.compile()
```

При входе в node `"subgraph"`:

- из родительского State берутся значения по ключам, общим с `SubgraphState` (в примере — `messages`), и передаются в подграф как начальное State.

- Подграф выполняется, обновляя своё State.

- По завершении подграфа, значения по общим ключам копируются обратно в родительское State (обновляя `messages`). Остальные ключи подграфа (`sub_custom`) не видны снаружи.


#### Преимущества subgraph <a id="subgraph-benefits"></a>
 
- Чистое разделение логики.

- Возможность тестировать подграф независимо.

- Упрощение больших графов.


#### Streaming

```python
# Режимы: values, updates, messages, custom, debug
async for chunk in graph.astream(
    {"messages": [HumanMessage(content="Tell me a story")]},
    config,
    stream_mode=["updates", "messages"]
):
    if "messages" in chunk:
        print(chunk["messages"], end="")
    if "updates" in chunk:
        print(f"\nUpdate: {chunk['updates']}")
```

#### Subgraph

```python
# Определяем подграф
subgraph_builder = StateGraph(MessagesState)
subgraph_builder.add_node("sub_node", lambda s: {"messages": [AIMessage(content="Subgraph processed")]})
subgraph_builder.set_entry_point("sub_node")
subgraph_builder.set_finish_point("sub_node")
subgraph = subgraph_builder.compile()

# Родительский граф с общим ключом "messages"
class ParentState(TypedDict):
    messages: Annotated[list, add_messages]
    parent_field: str

parent_builder = StateGraph(ParentState)
parent_builder.add_node("subgraph", subgraph)
parent_builder.add_node("final", lambda s: {"parent_field": "done"})
parent_builder.add_edge(START, "subgraph")
parent_builder.add_edge("subgraph", "final")
parent_builder.add_edge("final", END)
parent_graph = parent_builder.compile()

# При запуске messages автоматически передаются в подграф и обратно
```
---

## Module 6: Мультиагентные системы (Multi-Agent Systems)

Когда одно приложение становится слишком сложным, или задачи требуют разных экспертиз, на помощь приходят мультиагентные системы. Вместо одного «универсального» агента, мы создаем нескольких специализированных, которые координируют свою работу. LangGraph с его графовой архитектурой — идеальная среда для этого .

### 1. Базовые принципы координации <a id="module6-basic-principles"></a>

В мультиагентной системе каждый агент — это отдельный граф (или его часть), который имеет доступ к общему State или получает свою часть данных. Ключевой вопрос — **как агенты взаимодействуют**.


- **Shared State:** Все агенты читают и пишут в один общий словарь State. Это самый простой способ, но требует внимательного проектирования, чтобы агенты не «затирали» данные друг друга.

- **Каналы связи (Message Passing):** Агенты обмениваются сообщениями через специальные структуры в State. Один агент может отправить сообщение, а другой — прочитать и обработать его.


### 2. Паттерны мультиагентного взаимодействия <a id="module6-patterns"></a>

Существует несколько проверенных архитектурных паттернов для организации совместной работы агентов.


#### Supervisor (Агент-Супервизор) <a id="module6-supervisor"></a>

Это классический иерархический подход. Один главный агент (supervisor) получает задачу, анализирует её и решает, какому агенту-специалисту (worker) её делегировать. Supervisor также может собирать результаты и решать, завершена ли задача или нужно привлечь другого специалиста .

- **Когда использовать:** Когда задачи четко разделяются по domain-областям (например: аналитик, кодер, писатель) и нужен централизованный контроль.

- **Реализация в LangGraph:**

    1.  Создаем `Supervisor` граф.
    2.  Создаем отдельных агентов-воркеров (как узлы или как сабграфы).
    3.  Supervisor использует `conditional_edges` для маршрутизации задачи к нужному воркеру на основе своего решения (вызова LLM).
    4.  Воркер выполняется и возвращает результат в общее State.
    5.  Supervisor снова анализирует State и либо вызывает другого воркера, либо завершает работу.


#### Hierarchical Teams (Иерархические команды) <a id="module6-hierarchical-teams"></a>

Усложненная версия Supervisor, где воркеры сами могут быть супервизорами для своих подкоманд. Создается многоуровневая иерархия .


- **Пример:** Есть супервизор верхнего уровня «Проект». Он делегирует задачи команде «Исследование» (у которой свой супервизор и агенты для веб-поиска и анализа документов) и команде «Написание отчета» (с агентом-редактором и агентом-оформителем).


#### Multi-Agent Collaboration (Равноправное сотрудничество) <a id="module6-collaboration"></a>

Агенты работают как равные партнеры. Каждый агент может видеть результат работы другого и решать, нужно ли ему подключиться. Здесь нет централизованного контроллера, управление распределено .

- **Когда использовать:** Для задач, требующих тесного итеративного взаимодействия, например, когда исследователь находит данные, а визуализатор тут же строит график, и они могут обсуждать результаты.

- **Реализация в LangGraph:**
    - Создается цикл, в котором агенты вызываются по очереди или на основе событий. Один агент может положить свое сообщение в специальный канал (поле в state), на которое подписан другой агент.
    - Узлы могут иметь доступ к общему пулу инструментов .


### 3. Коммуникация между агентами <a id="module6-communication"></a>

- **Ручная маршрутизация:** Функция в `conditional_edge` анализирует State и решает, какой агент будет следующим.
- **Естественный язык:** Агенты пишут сообщения друг для друга на естественном языке в общее поле State (например, `messages`). Следующий агент читает эти сообщения и понимает контекст.




#### 13. Мультиагентные паттерны

#### Supervisor (один агент решает, кого вызвать)

```python
# Агенты-воркеры как узлы
def agent_a(state: MessagesState):
    return {"messages": [AIMessage(content="Result from A")]}

def agent_b(state: MessagesState):
    return {"messages": [AIMessage(content="Result from B")]}

def supervisor_router(state: MessagesState) -> Literal["agent_a", "agent_b", END]:
    # LLM вызывается для принятия решения
    if "done" in state["messages"][-1].content:
        return END
    return "agent_a"  # упрощённо

multi_builder = StateGraph(MessagesState)
multi_builder.add_node("agent_a", agent_a)
multi_builder.add_node("agent_b", agent_b)
multi_builder.add_conditional_edges(START, supervisor_router)
multi_builder.add_edge("agent_a", "supervisor_router")  # возвращаемся к супервизору
multi_builder.add_edge("agent_b", "supervisor_router")
multi_builder.set_finish_point(END)
multi_graph = multi_builder.compile()
```

---

## Module 7: Наблюдаемость и тестирование (Observability & Testing)

Разработка агента — это только полдела. Чтобы он надежно работал в продакшне, нужно понимать, что происходит у него «под капотом», и иметь систему для оценки его качества.

### 1. Observability (Наблюдаемость) с OpenTelemetry <a id="module7-observability"></a>

Стандартные метрики (CPU, RAM) ничего не скажут о качестве работы агента. Нужна информация о логике его выполнения. OpenTelemetry — отраслевой стандарт для сбора трассировок (traces), метрик и логов .

- **Зачем это нужно:**

    - **Трассировка выполнения:** Увидеть полный путь запроса: какие узлы и в каком порядке были вызваны, сколько времени занял каждый вызов LLM или инструмента .

    - **Анализ State:** Зафиксировать, как менялось State графа (`state`) до и после каждого узла. Это бесценно для отладки, когда агент пошел не по тому пути .

    - **Счетчики:** Сколько раз был вызван конкретный инструмент? Сколько токенов потрачено за запрос?

    - **Выявление узких мест:** Какой Node тормозит всю систему?

- **Как это интегрировать в LangGraph:**

    LangGraph, как и LangChain, поддерживает систему колбэков (callbacks). Вы можете создать свой `CallbackHandler`, который будет создавать `spans` OpenTelemetry при старте и завершении каждого узла .


    **Ключевые моменты для трассировки графа:**

    1.  **Корневой Span для всего запуска:** Создавайте один главный span для всего вызова `graph.invoke()`, чтобы группировать все дочерние операции .

    2.  **Span для каждого узла:** Для каждого выполнения узла создавайте отдельный span. В атрибуты span-а обязательно добавляйте :

        - Имя узла (`langgraph.node.name`).

        - Время выполнения (`langgraph.node.duration_ms`).

        - Размер State до и после (можно для начала просто количество ключей).

        - Количество измененных ключей.

        - **Важно:** Для узлов, которые могут вызываться несколько раз (например, в цикле), добавляйте счетчик посещений (`langgraph.node.visit_count`) .

    3.  **Conditional Edges:** Трассируйте, какое решение приняла функция роутинга и почему. Это можно сделать, записывая результат функции в атрибуты текущего span-а.

   **Инструментарий New Relic**: С февраля 2026 года New Relic добавил официальную поддержку инструментирования `@langchain/langgraph` . Это значит, что теперь можно получать готовые дашборды и метрики по работе ваших графов (латентность узлов, вызовы инструментов и т.д.) без написания кастомного колбэка. Это упрощает продакшн-мониторинг.

### 2. Тестирование и оценка качества (Evaluation, Evals) <a id="module7-evaluation"></a>
 

LLM-приложения недетерминированы, поэтому юнит-тесты здесь не панацея. Нужна система оценок (evals), которая будет проверять качество ответов.

**Тестирование (Evals)**: `langgraph-prebuilt 0.2.0` также принес значительное улучшение тестового покрытия (до 92%) и добавил структурированные тесты для HITL-логики . Для вас это сигнал, что фреймворк становится надежнее, и можно смелее использовать prebuilt-компоненты в критических сценариях.

- **Компоненты тестирования:**

    - **Тестовые наборы данных:** Наборы вопросов и эталонных ответов (или просто критериев).

    - **Метрики:**

        - **Программные:** Точность извлечения фактов, время ответа, количество вызовов инструментов.

        - **LLM-as-a-judge:** Использование сильной LLM (например, GPT-4) для оценки ответа по критериям: полезность, безопасность, соответствие фактам (отсутствие галлюцинаций) .

        - **Сравнение с эталоном:** Насколько ответ агента семантически близок к идеальному ответу.


- **LangSmith — платформа для всего цикла:**

    LangSmith — это платформа от создателей LangChain, которая неразрывно связана с LangGraph и предоставляет инструменты для каждого этапа .

    - **Трассировка:** Автоматически логирует каждый запуск вашего графа, позволяя смотреть, что происходило.

    - **Датасеты:** Создавайте датасеты для тестирования.

    - **Тестирование (Evals):** Запускайте тесты на этих датасетах и получайте отчеты с метриками. Можно запускать тесты при каждом изменении кода, чтобы убедиться, что новое изменение не сломало старую функциональность (регрессия).

    - **Мониторинг в проде:** Анализируйте трафик в реальном времени, отслеживайте латентность, затраты на токены и качество ответов .



#### Observability (Callbacks и LangSmith)

```python
# Простой callback для логирования
from langchain_core.callbacks import BaseCallbackHandler

class MyCallbackHandler(BaseCallbackHandler):
    def on_chain_start(self, serialized, inputs, **kwargs):
        print(f"Starting node: {serialized.get('name')}")
    
    def on_chain_end(self, outputs, **kwargs):
        print(f"Finished node, outputs: {outputs}")

# Подключаем при вызове
graph.invoke(inputs, config={"callbacks": [MyCallbackHandler()]})

# Для LangSmith просто установите переменные окружения:
# export LANGCHAIN_TRACING_V2=true
# export LANGCHAIN_API_KEY=...
```

---

## Module 8: Инструменты и Production (Tooling & Deployment)

*   **Stability и Production Readiness**: Выход версии 1.0 — это главное событие. Теперь команда LangChain обещает **никаких ломающих изменений до версии 2.0** . Это делает LangGraph идеальным выбором для долгосрочных корпоративных проектов. По данным опросов, 51% компаний уже используют агентов в проде, а 78% планируют . LangGraph — стандарт де-факто для этого.

*   **UV вместо pip**: Для управления зависимостями и сборки `langgraph-prebuilt` теперь используется инструмент `uv`, что значительно ускоряет разрешение зависимостей . Это техническая деталь для разработчиков, ее можно добавить в раздел про инструменты.


### 1. Prebuilt Components (Готовые компоненты) <a id="module8-prebuilt"></a>

Чтобы не писать с нуля типовые узлы, LangGraph предоставляет библиотеку `langgraph-prebuilt` .


- **`create_react_agent`** : Функция для создания классического ReAct-агента (Reasoning + Acting) в одну строку. Она создает граф, который в цикле вызывает LLM, выполняет инструменты и возвращает результат .

- **`ToolNode`** : Готовый Node, который умеет обрабатывать вызовы инструментов. На вход он принимает список инструментов и AI-сообщение с `tool_calls`, а на выходе возвращает результат выполнения инструментов .

- **`ValidationNode`** : Node для валидации аргументов инструментов на основе Pydantic-схемы. Если аргументы не проходят валидацию, он может сгенерировать сообщение об ошибке, чтобы LLM могла исправиться .

- **Инструменты для Human-in-the-Loop (`HumanInterrupt`)**: Prebuilt-библиотека также содержит удобные схемы и утилиты для организации прерываний с участием человека. Например, можно создать `HumanInterrupt`, который опишет задачу пользователю и предложит варианты ответа: проигнорировать, ответить, отредактировать .



### 2. Деплой в Production (LangGraph Platform) <a id="module8-langgraph-platform"></a>


LangChain предлагает готовую платформу для деплоя ваших агентов — LangGraph Platform . Это не единственный способ (можно развернуть и самостоятельно, например, на AWS ECS Fargate ), но самый интегрированный.


**LangGraph Platform** предоставляет :

- **1-click deploy из GitHub:** Забудьте про настройку серверов.

- **Встроенные чекпоинты и память:** Из коробки, без дополнительной настройки.

- **Масштабируемые API:** Автоматически создает REST API для вашего графа, поддерживает стриминг.

- **LangGraph Studio:** Визуальный интерфейс для отладки, где можно видеть граф, State, перематывать время и перезапускать с любого шага.

**Варианты деплоя:**
- **Cloud (SaaS):** Полностью управляемый облачный сервис.
- **Hybrid:** Управляющая панель в облаке, а данные и выполнение — в вашей инфраструктуре.
- **Self-Hosted:** Полностью на ваших серверах.



### 3. Альтернативный деплой: Самостоятельно на AWS <a id="module8-aws-deploy"></a>

Если по каким-то причинам LangGraph Platform не подходит, можно развернуть агента самостоятельно, используя современные практики DevOps. Хороший пример — шаблон для деплоя на AWS ECS Fargate .


**Ключевые компоненты такого решения:**

1.  **Контейнеризация:** Упаковываем FastAPI приложение с LangGraph агентом в Docker-образ.

2.  **Оркестрация:** Используем AWS ECS Fargate (бессерверные контейнеры) для запуска.

3.  **Балансировка:** Application Load Balancer (ALB) распределяет трафик.

4.  **Масштабирование:** Автоматическое масштабирование количества задач Fargate на основе нагрузки (CPU/Memory).

5.  **Безопасность:** Храним секреты (ключи API) в AWS Parameter Store или Secrets Manager .


#### Prebuilt компоненты

#### ReAct Agent (create_react_agent)

```python
from langchain_core.tools import tool

@tool
def get_weather(city: str):
    """Get weather for a city"""
    return f"Weather in {city} is sunny"

tools = [get_weather]
model = ChatOpenAI(model="gpt-4")
agent = create_react_agent(model, tools, checkpointer=MemorySaver())

response = agent.invoke({"messages": [HumanMessage(content="Weather in Paris?")]},
                         config={"configurable": {"thread_id": "thread_react"}})
```

#### ToolNode (для обработки вызовов инструментов)

```python
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools=[get_weather])
# tool_node принимает сообщение с tool_calls и возвращает результат
```

#### HumanInterruptNode (готовый Node для HITL)

```python
from langgraph.prebuilt import HumanInterruptNode

# В графе:
# builder.add_node("human_review", HumanInterruptNode())
# При достижении этого узла выполнение остановится и будет ждать Command(resume=...)
```

####  Простой деплой через FastAPI (скелет)

```python
# main.py
from fastapi import FastAPI
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver

app = FastAPI()
graph = builder.compile(checkpointer=MemorySaver())

@app.post("/invoke")
async def invoke(thread_id: str, message: str):
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke({"messages": [HumanMessage(content=message)]}, config)
    return result
```
---

## Уровни владения LangGraph <a id="lg-skills"></a>

| Уровень | Теоретические знания | Практические навыки | Типичные задачи / Проекты |
| :--- | :--- | :--- | :--- |
| **Junior** | - Понимает разницу между workflow и agent.<br>- Знает основные компоненты: State, Node, Edge, Conditional Edge.<br>- Знает базовые архитектурные паттерны (Prompt Chaining, Routing, Parallelization).<br>- Имеет представление о чекпоинтах и памяти (checkpoint, thread). | - Может создать простой линейный граф из 2–3 узлов.<br>- Умеет определять State через `TypedDict` и использовать простые редукторы (например, `add_messages`).<br>- Способен добавить conditional edge на основе простого условия (if/else).<br>- Может запустить граф с `MemorySaver` и получить историю сообщений в рамках одного треда.<br>- Умеет использовать `invoke` и простой `stream`. | - Чат-бот с памятью в рамках сессии.<br>- Граф, который вызывает LLM, затем инструмент (например, поиск погоды) и возвращает ответ.<br>- Реализация паттерна "Routing" (перенаправление запроса к разным промптам). |
| **Middle** | - Глубоко понимает механизм чекпоинтов и супер-шагов.<br>- Знает все режимы стриминга (`values`, `updates`, `messages`).<br>- Понимает разницу между short-term и long-term memory, знаком с `BaseStore` и `InMemoryStore`.<br>- Знает концепцию `interrupt` и `Command` для HITL.<br>- Освоил работу с `update_state` и `as_node` для ручной коррекции.<br>- Понимает, как работают `@task` и durable execution. | - Уверенно проектирует State с использованием редукторов и вложенных структур.<br>- Может интегрировать `store` для долговременной памяти, сохранять и извлекать данные о пользователе.<br>- Использует `interrupt` для создания точек подтверждения, обрабатывает возобновление с `Command`.<br>- Применяет `update_state` для отладки и изменения хода графа.<br>- Пишет простые тесты с использованием LangSmith или самодельных эвалюаций.<br>- Умеет комбинировать `@task` с вызовами API для обеспечения идемпотентности. | - Агент, который запоминает предпочтения пользователя между сессиями (long-term memory).<br>- Рабочий процесс с ручным подтверждением перед выполнением действия (например, отправка email).<br>- Интеграция с внешней БД через кастомный `Store`.<br>- Создание графа с несколькими точками прерывания и восстановления. |
| **Middle+** | - Знает продвинутые техники работы с памятью: trimming, summarization, управление длиной контекста.<br>- Понимает принципы Time-Travel (replay) и умеет использовать их для отладки и анализа.<br>- Знаком с `StateSchema` и новыми примитивами (`ReducedValue`, `UntrackedValue`, `MessagesValue`) из LangGraph 1.0.<br>- Понимает устройство subgraph и умеет проектировать композицию графов.<br>- Имеет представление о callback-системе и может писать кастомные обработчики для observability.<br>- Знает типовые проблемы при работе с interrupt (порядок, идемпотентность) и умеет их избегать. | - Реализует стратегии сокращения контекста (суммаризация, обрезка) в виде отдельных узлов.<br>- Использует `get_state_history` и replay для исследования альтернативных путей.<br>- Применяет `StateSchema` для более строгой типизации и оптимизации (через `UntrackedValue`).<br>- Создаёт переиспользуемые подграфы и включает их в родительские графы с общей типизацией.<br>- Пишет кастомные колбэки для отправки метрик в Prometheus или логирования в ELK.<br>- Проводит code review и помогает джуниорам избегать ошибок с interrupt. | - Агент с долгой историей диалога, который автоматически суммаризирует старые сообщения.<br>- Система, позволяющая откатиться к любому шагу и изменить решение (timetravel для отладки).<br>- Библиотека переиспользуемых подграфов для различных задач (поиск, анализ, генерация).<br>- Интеграция LangGraph с системой мониторинга (например, OpenTelemetry). |
| **Senior** | - Глубокое понимание мультиагентных архитектур: Supervisor, Hierarchical Teams, Collaborative agents.<br>- Знает, как проектировать общий State для нескольких агентов, каналы связи и синхронизацию.<br>- Владеет паттернами отказоустойчивости и распределённых систем применительно к агентам.<br>- Понимает внутреннее устройство LangGraph (как работают сабграфы, как оптимизировать производительность).<br>- Знает best practices для production: миграции БД, выбор чекпоинтера и store (Postgres, Redis), настройка индексов.<br>- Умеет проводить нагрузочное тестирование и оптимизировать графы. | - Проектирует и реализует сложные мультиагентные системы с иерархией и динамическим распределением задач.<br>- Настраивает production-хранилища (PostgresStore, RedisStore) с миграциями и резервным копированием.<br>- Оптимизирует графы: уменьшает количество супер-шагов, использует параллелизм с `Send`, минимизирует размер чекпоинтов.<br>- Пишет комплексные наборы тестов (unit, integration, eval) и интегрирует их в CI/CD.<br>- Проводит архитектурные ревью, выбирает подходящие паттерны под бизнес-задачи.<br>- Настраивает мониторинг и алертинг для агентов в проде. | - Мультиагентная система: супервизор, команда исследователей, команда писателей, агент-редактор.<br>- Миграция с in-memory на PostgreSQL в продакшене с нулевым даунтаймом.<br>- Проектирование системы с тысячами параллельных потоков (threads) и оптимизация стоимости.<br>- Внедрение LangGraph в крупном проекте, обучение команды. |
| **Expert** | - Вносит изменения в сам фреймворк (open source contributions) или создаёт расширения.<br>- Разрабатывает новые паттерны и подходы к построению агентов (например, рекурсивные агенты, динамические графы).<br>- Глубоко понимает теорию конечных автоматов, графы выполнения, детерминизм в контексте LLM.<br>- Исследует пересечение агентов и других областей (нейросимволические системы, reinforcement learning).<br>- Консультирует компании по внедрению агентных архитектур на LangGraph. | - Создаёт кастомные чекпоинтеры и сторы для специфических нужд (например, для работы с Kafka).<br>- Оптимизирует производительность на уровне C-расширений или асинхронных паттернов.<br>- Разрабатывает инструменты для визуализации, отладки и тестирования графов.<br>- Пишет статьи, выступает на конференциях, делится опытом с сообществом.<br>- Участвует в развитии экосистемы LangGraph (prebuilt, документация, примеры). | - Разработка нового типа узлов для фреймворка.<br>- Создание предметно-ориентированного языка для описания агентов на базе LangGraph.<br>- Исследовательский проект: рекурсивные агенты для обработки сверхдлинных контекстов.<br>- Стратегическое планирование использования агентов в масштабах всей компании. |